In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
df = pd.read_csv("../data/train.csv")
df_t = pd.read_csv("../data/test.csv")

In [3]:
df.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [4]:
test_ids = df_t["id"].copy()

df.drop(["id"], axis=1, inplace=True)
df_t.drop(["id"], axis=1, inplace=True)

In [5]:
for data in [df, df_t]:

    # Delay on Air
    data["Delay on air"] = (
        data["Arrival Delay in Minutes"]
        - data["Departure Delay in Minutes"]
    )

    # Average of Age, Gate Location and Delay on Air
    data["Age_Gate_Delay_Avg"] = (
        data["Age"]
        + data["Gate location"]
        + data["Delay on air"]
    ) / 3

    # Average of Ease of Online Booking and Online Boarding
    data["Online_Experience_Avg"] = (
        data["Ease of Online booking"]
        + data["Online boarding"]
    ) / 2

    # Average of Food and Drink and On-board Service
    data["Food_Service_Avg"] = (
        data["Food and drink"]
        + data["On-board service"]
    ) / 2

    # Average of Baggage Handling and Checkin Service
    data["Baggage_Checkin_Avg"] = (
        data["Baggage handling"]
        + data["Checkin service"]
    ) / 2

In [6]:
drop_cols = [
    "Gender",
    "Departure/Arrival time convenient",
    "Arrival Delay in Minutes",
    "Departure Delay in Minutes"
]

df.drop(columns=drop_cols, inplace=True)
df_t.drop(columns=drop_cols, inplace=True)

In [7]:
print("Train shape:", df.shape)
print("Test shape:", df_t.shape)

print("\nTrain columns:")
print(df.columns.tolist())

print("\nTest columns:")
print(df_t.columns.tolist())

Train shape: (699635, 23)
Test shape: (299844, 22)

Train columns:
['Age', 'Flight Distance', 'Inflight wifi service', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction', 'Delay on air', 'Age_Gate_Delay_Avg', 'Online_Experience_Avg', 'Food_Service_Avg', 'Baggage_Checkin_Avg']

Test columns:
['Age', 'Flight Distance', 'Inflight wifi service', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Customer Type', 'Type of Travel', 'Class', 'Delay on air', 'Age_Gate_Delay_Avg', 'Online_Experience_Avg', 'Food_Service_Avg', 'Baggage_Checkin_Avg']


In [8]:
import joblib
from sklearn.model_selection import train_test_split

# PREPROCESSING
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, FunctionTransformer 
from sklearn.impute import SimpleImputer

# MODELS
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# MODEL EVALUATION
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# HYPERPARAMETER TUNING
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

In [9]:
X = df.drop(['satisfaction'], axis = 1)
y = df['satisfaction']

In [10]:
num_cols = X.select_dtypes(include = ['int', 'float']).columns
cat_cols = X.select_dtypes(include='object').columns

In [11]:
left_skewed = []
right_skewed = []
for cols in num_cols:
    if df[cols].skew() > 0.5:
        print(f"{cols} is skewed with skewness: {df[cols].skew()}")
        right_skewed.append(cols)

    elif df[cols].skew() < -0.5:
        print(f"{cols} is negatively skewed with skewness: {df[cols].skew()}")
        left_skewed.append(cols)


print(f"Left skewed features: {left_skewed}")
print(f"Right skewed features: {right_skewed}")

Flight Distance is skewed with skewness: 1.0804656071734196
Seat comfort is negatively skewed with skewness: -0.6161920579994616
On-board service is negatively skewed with skewness: -0.5593974280113893
Baggage handling is negatively skewed with skewness: -0.8388565290421319
Delay on air is skewed with skewness: 2.10599987694359
Baggage_Checkin_Avg is negatively skewed with skewness: -0.5408097283204384
Left skewed features: ['Seat comfort', 'On-board service', 'Baggage handling', 'Baggage_Checkin_Avg']
Right skewed features: ['Flight Distance', 'Delay on air']


In [12]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline ([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore'))
])

In [13]:
preprocessor = ColumnTransformer ([
    ('num', numerical_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective='binary:logistic', 
                          eval_metric='logloss', random_state=42, n_jobs=-1))
])

In [14]:
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.2, random_state = 42, stratify=y)

In [15]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Age', 'Flight Distance', 'Inflight wifi service',
       'Ease of Online booking', 'Gate location', 'Food and drink',
       'Online boarding', 'Seat comfort', 'Inflight entertainment',
       'On-board service', 'Leg...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [16]:
prediction = pipeline.predict(X_test)

In [17]:
prediction_proba = pipeline.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, prediction_proba)
print(roc)

0.957371746349101


In [18]:
test_prediction = pipeline.predict_proba(df_t)[:, 1]

In [19]:
submission = pd.DataFrame({
    'id': test_ids,
    'target': test_prediction
})

submission.to_csv('submission.csv', index=False)

In [20]:
# Hyperparameter search space
param_distributions = {

    "xgb__n_estimators": [
        200, 300, 400, 500, 700
    ],

    "xgb__max_depth": [
        2, 3, 4, 5
    ],

    "xgb__learning_rate": [
        0.01, 0.02, 0.03, 0.05, 0.08
    ],

    "xgb__min_child_weight": [
        3, 5, 8, 10, 15
    ],

    "xgb__gamma": [
        0, 0.1, 0.2, 0.3, 0.5
    ],

    "xgb__subsample": [
        0.6, 0.7, 0.8, 0.9
    ],

    "xgb__colsample_bytree": [
        0.6, 0.7, 0.8, 0.9
    ],

    "xgb__reg_alpha": [
        0, 0.1, 0.3, 0.5, 1
    ],

    "xgb__reg_lambda": [
        1, 2, 3, 5, 8, 10
    ]
}

In [21]:
from sklearn.model_selection import StratifiedKFold

In [22]:
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True, 
    random_state = 42
 )

In [25]:
random_search = RandomizedSearchCV(
    estimator = pipeline,
    param_distributions = param_distributions,
    n_iter = 25, 
    scoring = 'roc_auc',
    cv = cv,
    random_state = 42,
    n_jobs = -1,
    verbose = 2,
    return_train_score = True
)

In [26]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='most_frequent')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               Index(['Age', 'Flight Distance', 'Inflight wifi service',
       'Ease of Online booking', 'Gate location', 'Fo...
                                        'xgb__gamma': [0, 0.1, 0.2, 0.3, 0.5],
                                        'xgb__learning_rate': [0.01, 0.02, 0.03,
                                                               0.05, 0.08],
                                        'xgb__max_depth': [2, 3, 4, 5],
                                        'xgb__min_child_weight': [3, 5, 8, 10,
                                                                  15],
                                        'xgb__n_estimators': [200, 300, 400,
                                                              500, 700],
                                        'xgb__reg_alpha': [0, 0.1, 0.3, 0.5, 1],
                                        'xgb__reg_lambda': [1, 2, 3, 5, 8, 10],
                                        'xgb__subsample': [0.6, 0.7, 0.8, 0.9]},
                   random_state=42, return_train_score=True, scoring='roc_auc',
                   verbose=2)

In [27]:
print("Best CV ROC-AUC:")
print(random_search.best_score_)

print("\nBest Parameters:")
print(random_search.best_params_)

Best CV ROC-AUC:
0.9574729193840451

Best Parameters:
{'xgb__subsample': 0.8, 'xgb__reg_lambda': 3, 'xgb__reg_alpha': 0.1, 'xgb__n_estimators': 400, 'xgb__min_child_weight': 10, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.08, 'xgb__gamma': 0.3, 'xgb__colsample_bytree': 0.9}


In [28]:
best_model = random_search.best_estimator_

test_proba = best_model.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(
    y_test,
    test_proba
)

print("Holdout ROC-AUC:", test_auc)

Holdout ROC-AUC: 0.9564790696366274


In [29]:
train_proba = best_model.predict_proba(X_train)[:, 1]

train_auc = roc_auc_score(
    y_train,
    train_proba
)

print("Train ROC-AUC:", train_auc)
print("Test ROC-AUC :", test_auc)
print("Difference   :", train_auc - test_auc)

Train ROC-AUC: 0.9589326208905146
Test ROC-AUC : 0.9564790696366274
Difference   : 0.002453551253887132


In [30]:
final_model = random_search.best_estimator_
final_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Age', 'Flight Distance', 'Inflight wifi service',
       'Ease of Online booking', 'Gate location', 'Food and drink',
       'Online boarding', 'Seat comfort', 'Inflight entertainment',
       'On-board service', 'Leg...
                               feature_types=None, feature_weights=None,
                               gamma=0.3, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.08,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=10, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=400, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [31]:
prediction_proba_tuned = final_model.predict_proba(df_t)[:, 1]

In [33]:
submission2 = pd.DataFrame({
    'id': test_ids,
    'target': test_prediction
})

submission2.to_csv('submission2.csv', index=False)

In [34]:
from catboost import CatBoostClassifier
X_cat = X.copy()
X_test_cat = df_t.copy()

# Find categorical columns automatically
cat_cols_catboost = X_cat.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(cat_cols_catboost)

Categorical columns:
['Customer Type', 'Type of Travel', 'Class']


In [36]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_oof = np.zeros(len(X_cat))
cat_test_pred = np.zeros(len(X_test_cat))

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_cat, y)
):

    print(f"\n===== Fold {fold + 1} =====")

    X_tr = X_cat.iloc[train_idx]
    X_val = X_cat.iloc[valid_idx]

    y_tr = y[train_idx]
    y_val = y[valid_idx]

    model_cat = CatBoostClassifier(
        iterations=1500,
        depth=7,
        learning_rate=0.03,

        loss_function="Logloss",
        eval_metric="AUC",

        l2_leaf_reg=5,
        random_strength=1,

        bootstrap_type="Bernoulli",
        subsample=0.8,

        random_seed=42,
        verbose=200,

        thread_count=-1
    )

    model_cat.fit(
        X_tr,
        y_tr,

        cat_features=cat_cols_catboost,

        eval_set=(X_val, y_val),

        early_stopping_rounds=150,

        verbose=200
    )

    # Validation probabilities
    val_pred = model_cat.predict_proba(X_val)[:, 1]

    cat_oof[valid_idx] = val_pred

    # Test probabilities
    cat_test_pred += (
        model_cat.predict_proba(X_test_cat)[:, 1]
        / cv.n_splits
    )

    fold_auc = roc_auc_score(
        y_val,
        val_pred
    )

    fold_scores.append(fold_auc)

    print("Fold ROC-AUC:", fold_auc)


===== Fold 1 =====
0:	test: 0.9415438	best: 0.9415438 (0)	total: 505ms	remaining: 12m 36s
200:	test: 0.9559556	best: 0.9559556 (200)	total: 1m 26s	remaining: 9m 16s
400:	test: 0.9572069	best: 0.9572069 (400)	total: 2m 49s	remaining: 7m 43s
600:	test: 0.9577004	best: 0.9577034 (599)	total: 4m 5s	remaining: 6m 6s
800:	test: 0.9579488	best: 0.9579488 (800)	total: 5m 19s	remaining: 4m 38s
1000:	test: 0.9581471	best: 0.9581471 (1000)	total: 6m 33s	remaining: 3m 16s
1200:	test: 0.9582864	best: 0.9582864 (1200)	total: 7m 47s	remaining: 1m 56s
1400:	test: 0.9583699	best: 0.9583699 (1400)	total: 9m 1s	remaining: 38.3s
1499:	test: 0.9584047	best: 0.9584047 (1499)	total: 9m 38s	remaining: 0us

bestTest = 0.9584046936
bestIteration = 1499

Fold ROC-AUC: 0.958404693578918

===== Fold 2 =====
0:	test: 0.9400329	best: 0.9400329 (0)	total: 360ms	remaining: 9m
200:	test: 0.9547162	best: 0.9547162 (200)	total: 1m 13s	remaining: 7m 57s
400:	test: 0.9558543	best: 0.9558543 (400)	total: 2m 31s	remaining: 

In [37]:
print("\nFold scores:")
print(fold_scores)

print("\nMean CV ROC-AUC:")
print(np.mean(fold_scores))

print("\nSTD:")
print(np.std(fold_scores))

print("\nOOF ROC-AUC:")
print(
    roc_auc_score(
        y,
        cat_oof
    )
)


Fold scores:
[np.float64(0.958404693578918), np.float64(0.9571224279977865), np.float64(0.95901489169405), np.float64(0.9582747028259236), np.float64(0.9587128792427466)]

Mean CV ROC-AUC:
0.9583059190678849

STD:
0.0006449249164623159

OOF ROC-AUC:
0.958298053483221


In [39]:
submission_cat = pd.DataFrame({
    "id": test_ids,
    "satisfaction": cat_test_pred
})

submission_cat.to_csv(
    "../data/submission_catboost.csv",
    index=False
)

submission_cat.head()

,id,satisfaction
0,699635,0.282929
1,699636,0.887808
2,699637,0.883548
3,699638,0.060274
4,699639,0.060234
